In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/home/xiaowenz/finetune')

In [2]:
from PIL import Image
from pathlib import Path
import datasets

def convert_to_swift_format(conversation, media_dir):
  """
  Convert a conversation from the current format to Swift format.

  Args:
      conversation: List of message dictionaries in the format:
          [{'role': 'system', 'content': '...'},
           {'role': 'user', 'content': [{'video': 'path/to/video.mp4'}]},
           {'role': 'user', 'content': 'Question text'},
           {'role': 'assistant', 'content': 'Answer tdext'}]

  Returns:
      Dictionary in Swift format with 'messages' key and optional media keys
  """
  result = {"messages": []}
  images = []
  videos = []
  audios = []

  # Track media placeholders for content substitution
  image_count = 0
  video_count = 0
  audio_count = 0

  for message in conversation:
    role = message['role']
    content = message['content']

    # Process content based on its type
    if isinstance(content, str):
      # Simple text content
      messages = result['messages']
      if messages and messages[-1]['role'] == role:
        messages[-1]['content'] += '\n' + content
      else:
        result["messages"].append({
            "role": role,
            "content": content
        })
    elif isinstance(content, list):
      # Content with potential media
      text_parts = []

      for item in content:
        if isinstance(item, dict):
          if 'text' in item:
            text_parts.append(item['text'])
          elif 'image' in item:
            text_parts.append("<image>")
            img = item['image']
            if isinstance(img, Image.Image):
              # Save the image to the media directory
              image_path = media_dir / f"image_{image_count}.png"
              img.save(image_path)
              img = image_path
            images.append(img)
            image_count += 1
          elif 'video' in item:
            text_parts.append("<video>")
            videos.append(item['video'])
            video_count += 1
          else:
            raise ValueError(f"Unsupported media type in item: {item}")
        elif isinstance(item, str):
          text_parts.append(item)

      # Join all text parts
      content_text = ''.join(text_parts)
      messages = result['messages']
      if messages and messages[-1]['role'] == role:
        messages[-1]['content'] += '\n' + content_text
      else:
        result["messages"].append({
            "role": role,
            "content": content_text
        })

  # Add media arrays if they exist
  if images:
    result["images"] = images
  if videos:
    result["videos"] = videos
  if audios:
    result["audios"] = audios

  return result


def convert_conversations_to_swift_format(conversations: list[dict], media_dir: str | Path) -> list[dict]:
  """
  Convert a list of conversations to Swift format.

  Args:
      conversations: List of conversations

  Returns:
      List of dictionaries in Swift format
  """
  n = len(conversations)
  z_fill = lambda x: str(x).zfill(len(str(n)))
  media_dir = Path(media_dir)
  return [
    convert_to_swift_format(conv, media_dir / z_fill(i))
    for i, conv in enumerate(conversations)
  ]


def convert_datasetdict(conv_dict: dict[str, list[dict]], media_dir: str | Path) -> dict[str, list[dict]]:
  """
  Convert a dataset dictionary to Swift format.

  Args:
      conv_dict: Dictionary with keys as conversation IDs and values as lists of messages

  Returns:
      Dictionary in Swift format
  """
  media_dir = Path(media_dir)
  return {
      k: convert_conversations_to_swift_format(v, media_dir / k)
      for k, v in conv_dict.items()
  }

In [3]:
from transformers import AutoProcessor
from qwenvl.argument import ProcessingArguments
from qwenvl.data import avail_datasets
from qwenvl.data.conversation import *
from qwenvl.data.preprocess import VerifyMediaStrategy, GetNumMediaTokensStrategy, GetNumTokensStrategy
from qwenvl.data.prompts import SYS_PROMPTS, USR_PROMPTS
from qwenvl.utils import get_logger
import datasets

logger = get_logger(__name__)
def create_swift_dataset(
    ds_name: str,
    split: str,
    modifiers: list[ConversationModifier] = [],
    **kwargs
):
  ds_config = avail_datasets[ds_name]
  print(ds_config)
  ds = datasets.load_dataset(ds_config['ds_key'], split=split)
  print(ds)
  ids = [i for i in range(len(ds))]
  base_cm = ds_config['cm'](
      **ds_config,
      **kwargs
  )
  
  cp = ConversationProcessor(
      conversation_maker=base_cm,
      conversation_modifiers=modifiers,
      **ds_config
  )
  
  if 'fashion' not in ds_name:
    ds = VerifyMediaStrategy(cp.get_content, None)(ds)
    answers = ds['answer']
  else:
    answers = ds.features['label'].int2str(ds['label'])
  processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", use_fast=True)
  proc_args = ProcessingArguments(
      image_min_pixels=processor.image_processor.min_pixels,
      image_max_pixels=processor.image_processor.max_pixels,
      video_min_pixels=processor.video_processor.min_pixels,
      video_max_pixels=processor.video_processor.max_pixels,
      base_interval=1
  )
  ds = GetNumMediaTokensStrategy(cp.get_content, proc_args)(ds)
  length = ds['num_media_tokens']
  convos = [cp(item) for item in ds]
  logger.info(f"Processed {len(convos)} items in split '{split}'")

  idx = 0
  logger.info(f"Pre conversion: {convos[idx]}")

  ds = convert_conversations_to_swift_format(convos, ds_config['media_dir'])
  ds = datasets.Dataset.from_list(ds).add_column('length', length).add_column('label', answers).add_column('id', ids).shuffle()

  logger.info(f"Post conversion: {ds[idx]}")
  return ds

In [4]:
def create_swift_w_length(ds_name, split, for_training):
  ds_config = avail_datasets[ds_name]
  processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", use_fast=True)
  proc_args = ProcessingArguments(
      image_min_pixels=processor.image_processor.min_pixels,
      image_max_pixels=processor.image_processor.max_pixels,
      video_min_pixels=processor.video_processor.min_pixels,
      video_max_pixels=processor.video_processor.max_pixels,
      base_interval=1,
  )
  ds = datasets.load_dataset(ds_config['ds_key'], split=split)
  base_cm = ds_config['cm'](**ds_config)
  ds = VerifyMediaStrategy(base_cm.get_content, proc_args)(ds)
  ds = GetNumMediaTokensStrategy(base_cm.get_content, proc_args)(ds)
  num_med_map = dict(zip(ds['video'], ds['num_media_tokens']))
  videos, qa_pairs, lengths = [], [], []
  for item in ds:
    if 'qa_pairs' not in item:
      item['qa_pairs'] = [{'question': item['question'], 'answer': item['answer']}]
    for qa_pair in item['qa_pairs']:
      videos.append(os.path.join(ds_config['media_dir'], item['video']))
      qa_pairs.append(qa_pair)
      lengths.append(num_med_map[item['video']])
  new_ds = datasets.Dataset.from_dict({
    'id': list(range(len(qa_pairs))),
    'video': videos,
    'qa_pairs': qa_pairs,
    'length': lengths,
  })
  ds_config['qa_list_field'] = 'qa_pairs'
  cm = VQACM(**ds_config, for_training=for_training)
  convos = [cm(item) for item in new_ds]
  swift = convert_conversations_to_swift_format(convos, ds_config['media_dir'])
  swift_ds = datasets.Dataset.from_list(swift).add_column('length', lengths).add_column(
    'label', [i['answer'] for i in qa_pairs]).add_column('id', new_ds['id']).shuffle()

  return swift_ds

train = create_swift_w_length('openbiomedvid_qa', 'train', True)
test = create_swift_w_length('surgeryvid', 'test', False)
train = datasets.DatasetDict({'train': train})
test = datasets.DatasetDict({'train': test})
train.push_to_hub("withcomment/surgeryvid_train")
test.push_to_hub("withcomment/surgeryvid_test")

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


2025-08-11 11:00:47,855 - qwenvl.data.preprocess - INFO - Counting media tokens in the dataset.
2025-08-11 11:00:48,010 - qwenvl.data.preprocess - INFO - Average number of media tokens: 9700.96


Verifying media content (num_proc=32):   0%|          | 0/2692 [00:00<?, ? examples/s]

2025-08-11 11:01:01,170 - qwenvl.data.preprocess - ERROR - Video file /scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/DVtJB2rTFcI_24_312.mp4 has zero frames or zero FPS with OpenCV.
2025-08-11 11:01:01,173 - qwenvl.data.preprocess - ERROR - Video file /scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/DVtJB2rTFcI_24_312.mp4 has zero frames or zero FPS with OpenCV.
2025-08-11 11:01:01,174 - qwenvl.data.preprocess - ERROR - Video file /scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/DVtJB2rTFcI_24_312.mp4 has zero frames or zero FPS with OpenCV.
2025-08-11 11:01:01,175 - qwenvl.data.preprocess - ERROR - Video file /scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/DVtJB2rTFcI_24_312.mp4 has zero frames or zero FPS with OpenCV.
2025-08-11 11:01:01,176 - qwenvl.data.preprocess - ERROR - Video file /scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/DVtJB2rTFcI_24_312.mp4 has zero frames or zero FPS with OpenCV.
2025-08-11 11:01:01,177 - qwenvl.data.preproc

Counting media tokens (num_proc=32):   0%|          | 0/2650 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/79 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/491 [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/withcomment/surgeryvid_test/commit/1c0f4f8571451af22a0a9b26ad70d4175555d3f8', commit_message='Upload dataset', commit_description='', oid='1c0f4f8571451af22a0a9b26ad70d4175555d3f8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/withcomment/surgeryvid_test', endpoint='https://huggingface.co', repo_type='dataset', repo_id='withcomment/surgeryvid_test'), pr_revision=None, pr_num=None)

In [5]:
'length' in test['train'].features

True